# Assistants 

[Assistants](https://docs.langchain.com/langsmith/assistants) give developers a quick and easy way to modify and version agents for experimentation.

## Supplying configuration to the graph

Our `task_maistro` graph is already set up to use assistants!

It has a `configuration.py` file defined and loaded in the graph.

We access configurable fields (`user_id`, `todo_category`, `task_maistro_role`) inside the graph nodes.

## Creating assistants 

Now, what is a practical use case for assistants with the `task_maistro` app that we've been building?

For me, it's the ability to have separate ToDo lists for different categories of tasks. 

For example, I want one assistant for my personal tasks and another for my work tasks.

These are easily configurable using the `todo_category` and `task_maistro_role` configurable fields.

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

This is the default assistant that we created when we deployed the graph.

In [1]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### Personal assistant

This is the personal assistant that I'll use to manage my personal tasks.

In [2]:
personal_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": {"todo_category": "personal"}}
)
print(personal_assistant)

{'assistant_id': '664df4f9-e535-42d5-b3ec-ec5ca5a3e395', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-09-23T18:34:55.858375+00:00', 'updated_at': '2026-09-23T18:34:55.858375+00:00', 'config': {'configurable': {'todo_category': 'personal'}}, 'context': {'todo_category': 'personal'}, 'metadata': {}, 'name': 'Untitled', 'description': None}


Let's update this assistant to include my `user_id` for convenience,  [creating a new version of it](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant). 

In [3]:
task_maistro_role = """You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:

- Help track and organize personal tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- Proactively ask for deadlines when new tasks are added without them
- Maintain a supportive tone while helping the user stay accountable
- Help prioritize tasks based on deadlines and importance

Your communication style should be encouraging and helpful, never judgmental. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Would you like to add one to help us track it better?"""

configurations = {"todo_category": "personal", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)
print(personal_assistant)

{'assistant_id': '664df4f9-e535-42d5-b3ec-ec5ca5a3e395', 'graph_id': 'task_maistro', 'version': 2, 'created_at': '2026-09-23T18:34:55.858375+00:00', 'updated_at': '2026-09-23T18:37:25.705000+00:00', 'config': {'configurable': {'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style should be encouraging and helpful, never judgmental. \n\

### Work assistant

Now, let's create a work assistant. I'll use this for my work tasks.

In [4]:
task_maistro_role = """You are a focused and efficient work task assistant. 

Your main focus is helping users manage their work commitments with realistic timeframes. 

Specifically:

- Help track and organize work tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:
  • Developer Relations features: typically 1 day
  • Course lesson reviews/feedback: typically 2 days
  • Documentation sprints: typically 3 days
- Help prioritize tasks based on deadlines and team dependencies
- Maintain a professional tone while helping the user stay accountable

Your communication style should be supportive but practical. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Based on similar tasks, this might take [suggested timeframe]. Would you like to set a deadline with this in mind?"""

configurations = {"todo_category": "work", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

work_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": configurations}
)
print(work_assistant)

{'assistant_id': '5fb9aa9a-b522-40b3-8f87-19194f48ea61', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-09-23T18:37:27.947885+00:00', 'updated_at': '2026-09-23T18:37:27.947885+00:00', 'config': {'configurable': {'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks base

## Using assistants 

Assistants will be saved to `Postgres` in our deployment.  

This allows us to easily search <!--[~search~](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)--> [search](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search) for assistants with the SDK.

In [5]:
assistants = await client.assistants.search()
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],
        'version': assistant['version'],
        'config': assistant['config']
    })

{'assistant_id': '5fb9aa9a-b522-40b3-8f87-19194f48ea61', 'version': 1, 'config': {'configurable': {'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks based on deadlines and team dependencies\n- Maintain a professional tone while helping the user stay accountable\n\nYour communicati

We can manage them easily with the SDK. For example, we can delete assistants that we're no longer using.  
> The syntax in the video is slightly off. The updated code below creates a spare assistant and then deletes it. 

In [6]:
# create a temporary assitant
temp_assistant = await client.assistants.create(
    "task_maistro", 
    config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"before delete: {{'assistant_id': {assistant['assistant_id']}}}")
    
# delete our temporary assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"after delete: {{'assistant_id': {assistant['assistant_id']} }}")

before delete: {'assistant_id': d8df2f57-c043-441e-b61a-b789c752351c}
before delete: {'assistant_id': 5fb9aa9a-b522-40b3-8f87-19194f48ea61}
before delete: {'assistant_id': 664df4f9-e535-42d5-b3ec-ec5ca5a3e395}
before delete: {'assistant_id': ea4ebafa-a81d-5063-a5fa-67c755d98a21}

after delete: {'assistant_id': d8df2f57-c043-441e-b61a-b789c752351c }
after delete: {'assistant_id': 5fb9aa9a-b522-40b3-8f87-19194f48ea61 }
after delete: {'assistant_id': 664df4f9-e535-42d5-b3ec-ec5ca5a3e395 }


Let's set the assistant IDs for the `personal` and `work` assistants that I'll work with.

In [7]:
work_assistant_id = assistants[0]['assistant_id']
personal_assistant_id = assistants[1]['assistant_id']

### Work assistant

Let's add some ToDos for my work assistant.

In [8]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_106543)
 Call ID: call_106543
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'status': 'not started', 'solutions': [], 'deadline': '2026-09-23', 'task': 'Re-film Module 6, lesson 5', 'time_to_complete': None}

New ToDo created:
Content: {'deadline': '2026-09-28', 'task': 'Update audioUX', 'status': 'not started', 'solutions': [], 'time_to_complete': None}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have updated your ToDo list with those tasks!', 'extras': {'signature': 'El4KXAFpFH0T/hZQKYxXGvSB6948JcSGatkuJ8qPwjTi6c+ViDZoGuXs

In [9]:
user_input = "Create another ToDo: Finalize set of report generation tutorials."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create another ToDo: Finalize set of report generation tutorials.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_84775)
 Call ID: call_84775
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'deadline': None, 'solutions': [], 'task': 'Finalize set of report generation tutorials', 'status': 'not started', 'time_to_complete': 60}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have updated your ToDo list to include "Finalize set of report generation tutorials".', 'extras': {'signature': 'El4KXAFpFH0TPP/gBo7zKmrWckgdRqrjcXipiuCa28BlYeIj6AozZjIkW6A2iZJy1L+xQ39Gw20I2+1/kVaIaOc4wk8xLE7NlkxQZ67iw0NnUwCH9HRfpxGjig2zShFa'}}]


The assistant uses it's instructions to push back with task creation! 

It asks me to specify a deadline :) 

In [10]:
user_input = "OK, for this task let's get it done by next Tuesday."
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

OK, for this task let's get it done by next Tuesday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_186333)
 Call ID: call_186333
  Args:
    update_type: todo
================================= Tool Message =================================

Document 82ff40fa-baf4-4c5e-810a-2cbb922995b5 updated:
Plan: Set the deadline of the 'Finalize set of report generation tutorials' task to next Tuesday (2026-09-29T00:00:00).
Added content: 2026-09-29T00:00:00
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have updated your ToDo list with the deadline for "Finalize set of report generation tutorials" set to next Tuesday, September 29, 2026.', 'extras': {'signature': 'El4KXAFpFH0Te/qFR0o7VhDRihvhpQIts/6AFGuWO8SxCeLDVaGqg/2T6u4d28M6/rVqUKxWPJhVSGGos1VOI0q0+5md49LnnitOv+rL+K2owFQN2TyOiFbOGsHc

### Personal assistant

Similarly, we can add ToDos for my personal assistant.

In [11]:
user_input = "Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_89017)
 Call ID: call_89017
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Check on swim lessons for the baby this weekend', 'time_to_complete': 30, 'solutions': ['Search local swim schools or community pools for weekend baby swim classes'], 'deadline': '2026-09-27T00:00:00', 'status': 'not started'}

New ToDo created:
Content: {'time_to_complete': 15, 'deadline': '2026-10-01T00:00:00', 'solutions': ['Log into American Express account and check available membership rewards points for winter travel'], 'task': 'For winter travel, check AmEx points', 'status': 'not started'}
================

In [12]:
user_input = "Give me a todo summary."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Give me a todo summary.
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Here is a summary of your current ToDo list, ordered by deadline:\n\n1. **Re-film Module 6, lesson 5**\n   - **Deadline:** September 23, 2026\n   - **Status:** Not started\n   - **Estimated Time:** 120 minutes\n   - **Solution:** Set up camera, review script, and record video\n\n2. **Check on swim lessons for the baby this weekend**\n   - **Deadline:** September 27, 2026\n   - **Status:** Not started\n   - **Estimated Time:** 30 minutes\n   - **Solution:** Search local swim schools or community pools for weekend baby swim classes\n\n3. **Update audioUX**\n   - **Deadline:** September 28, 2026\n   - **Status:** Not started\n   - **Estimated Time:** 60 minutes\n   - **Solution:** Review and update audioUX assets and implementation\n\n4. **Finalize set of report generation tuto